# Demo: borrador de email ante incidencia de compras

Flujo: ver las incidencias -> elegir un lote y escribir un comentario -> ver la incidencia, la evaluacion de la IA sobre el historial del proveedor, el prompt (plantilla editable + prompt final) y el borrador generado.

Cada celda se puede volver a ejecutar de forma independiente tras cambiar las variables marcadas con `# <-- editar`.

In [1]:
import sys
from datetime import date
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src.config import INCIDENCIAS_CSV_PATH
from src.historial import calcular_historial_proveedor, formatear_evaluacion_historico
from src.generador import construir_prompt_completo
from src.prompts import PLANTILLA_BORRADOR
from src.validador_borrador import comprobar_numeros_permitidos
from src.config import LLM_MODEL, LLM_TEMPERATURE, LLM_TOP_K, LLM_TOP_P
from langchain_ollama import ChatOllama

## 1. Incidencias disponibles

In [2]:
df = pd.read_csv(INCIDENCIAS_CSV_PATH)
df

,proveedor,lote,tiempo_compra,precio_compra,tipo,incidencia
0,B,L-1535,2026-07-06,1.20,manzana,documentacion_incorrecta
1,A,L-3811,2026-05-15,1.04,naranja,cadena_frio
2,A,L-3257,2026-06-21,1.98,ciruela,cadena_frio
3,A,L-3611,2026-04-30,2.26,uva,cadena_frio
4,A,L-0106,2026-02-15,2.69,kiwi,cadena_frio
...,...,...,...,...,...,...
185,P,L-0803,2026-03-13,2.77,kiwi,variedad_incorrecta
186,P,L-3770,2026-03-14,4.00,aguacate,calidad_producto
187,G,L-2240,2026-04-05,0.59,naranja,producto_danado
188,M,L-7222,2024-08-04,1.08,manzana,calidad_producto


## 2. Elige un lote y escribe tu comentario (input del comercial)

In [3]:
LOTE = "L-7222"  # <-- editar: elige un lote de la tabla de arriba
COMENTARIO_COMERCIAL = ""  # <-- editar: tu comentario libre para incluir en el email

fila = df[df["lote"] == LOTE].iloc[0]
proveedor = fila["proveedor"]
lote = fila["lote"]
fecha = fila["tiempo_compra"]
incidencia_actual = fila["incidencia"]

fila.to_frame().T

,proveedor,lote,tiempo_compra,precio_compra,tipo,incidencia
188,M,L-7222,2024-08-04,1.08,manzana,calidad_producto


## 3. Evaluacion de la IA sobre el historial del proveedor

In [4]:
historial = calcular_historial_proveedor(df, proveedor, fecha_referencia=date.today().isoformat())
evaluacion_historico = formatear_evaluacion_historico(historial)

print(evaluacion_historico)
historial

Este proveedor registra 15 incidencia(s) en total, de las cuales 3 se han producido recientemente. La incidencia mas frecuente es 'calidad_producto'. La tasa de reincidencia en la misma categoria es del 66.67%. La gravedad media de sus incidencias es 3.4.


{'num_incidencias_total': 15,
 'num_incidencias_ultimos_6_meses': 3,
 'incidencia_mas_frecuente': 'calidad_producto',
 'tasa_reincidencia': 66.67,
 'gravedad_media': np.float64(3.4)}

## 4. Tono y objetivo deseados

In [ ]:
TONO = "firme-pero-cordial"  # <-- editar: formal / firme-pero-cordial / conciliador
OBJETIVO = "una nueva incidencia generaría un insumplimiento grave en el contrato"  # <-- editar: pedir explicacion / exigir compensacion / avisar / cerrar incidencia


#El modelo es tan fiel que una falta otográfica en el objetivo se pasa al
#correo

## 5. Plantilla del prompt (editable)

Si necesitas retocar el prompt a mano, edita el texto de `mi_plantilla` (debe conservar los placeholders `{proveedor}`, `{lote}`, `{fecha}`, `{incidencia_actual}`, `{evaluacion_historico}`, `{ejemplos_rag}`, `{tono}`, `{objetivo}`, `{mensaje_libre}`) y vuelve a ejecutar esta celda y las siguientes.

In [6]:
mi_plantilla = PLANTILLA_BORRADOR  # <-- editar aqui si quieres retocar el prompt
print(mi_plantilla)

Eres un asistente del departamento de compras 
de una empresa de fruta. Tu tarea es redactar un borrador de email 
de respuesta a un proveedor ante una incidencia, en espanol, 
listo para revision humana.

## Datos de la incidencia actual
- Proveedor: {proveedor}
- Lote: {lote}
- Fecha: {fecha}
- Categoria de incidencia: {incidencia_actual}

## Evaluacion del historial del proveedor
{evaluacion_historico}

## Ejemplos de emails similares ya redactados (para inspirar el estilo, no copiar literalmente)
{ejemplos_rag}

## Instrucciones del usuario
- Tono deseado: {tono}
- Objetivo del email: {objetivo}
- Mensaje libre a incluir (si aplica): {mensaje_libre}

## Requisitos del borrador
- Estructura: asunto + cuerpo (saludo, referencia al lote/fecha,
descripcion de la incidencia, mencion al historial si es relevante,
peticion/objetivo si lo hay, cierre).
- Cuando menciones el proveedor, el lote o la fecha, copia literalmente el
valor indicado arriba (ej.: "lote {lote}"). No escribas placehol

## 6. Prompt final ensamblado (con los datos e ejemplos RAG ya insertados)

In [7]:
prompt_final = construir_prompt_completo(
    proveedor=proveedor,
    lote=lote,
    fecha=fecha,
    incidencia_actual=incidencia_actual,
    historial=historial,
    tono=TONO,
    objetivo=OBJETIVO,
    mensaje_libre=COMENTARIO_COMERCIAL or None,
    plantilla=mi_plantilla,
)
print(prompt_final)

Eres un asistente del departamento de compras 
de una empresa de fruta. Tu tarea es redactar un borrador de email 
de respuesta a un proveedor ante una incidencia, en espanol, 
listo para revision humana.

## Datos de la incidencia actual
- Proveedor: M
- Lote: L-7222
- Fecha: 2024-08-04
- Categoria de incidencia: calidad_producto

## Evaluacion del historial del proveedor
Este proveedor registra 15 incidencia(s) en total, de las cuales 3 se han producido recientemente. La incidencia mas frecuente es 'calidad_producto'. La tasa de reincidencia en la misma categoria es del 66.67%. La gravedad media de sus incidencias es 3.4.

## Ejemplos de emails similares ya redactados (para inspirar el estilo, no copiar literalmente)
(producto_danado, firme-pero-cordial): Buenos días,

Tras la recepción del lote L-3386, correspondiente al envío del 2026-04-18 por parte de Frutales Vega, hemos comprobado que parte de la mercancía llegó dañada durante el transporte. Confiamos en la buena colaboración m

## 7. Borrador generado por el LLM

In [8]:
llm = ChatOllama(model=LLM_MODEL, temperature=LLM_TEMPERATURE, top_k=LLM_TOP_K, top_p=LLM_TOP_P)
borrador = llm.invoke(prompt_final).content
print(borrador)

**Asunto:** Re: Incidencia en Lote L-7222 del 2024-08-04 - Calidad del Producto  

Buenos días,  

Tras la recepción del lote L-7222, correspondiente al envío del 2024-08-04 por parte de M, hemos comprobado que parte de la mercancía presenta problemas de calidad, lo cual afecta la conformidad del pedido. Este proveedor registra un historial de 15 incidencias en total, de las cuales 3 se han producido recientemente, y la categoría "calidad_producto" es la más frecuente, con una tasa de reincidencia del 66.67%. La gravedad media de sus incidencias es 3.4, lo que subraya la importancia de abordar esta situación con urgencia.  

Dada la frecuencia de este tipo de incidencias y su impacto en el cumplimiento del contrato, solicitamos una respuesta inmediata para resolver esta situación y evitar un insumplimiento grave. Confiamos en la colaboración para garantizar la continuidad del suministro y mantener una relación de trabajo satisfactoria.  

Un saludo,  
Departamento de Compras


## 8. Verificacion anti-alucinacion de cifras (opcional)

In [ ]:
numeros_permitidos = {
    str(historial["num_incidencias_total"]),
    str(historial["num_incidencias_ultimos_6_meses"]),
    str(historial["tasa_reincidencia"]),
    str(historial["gravedad_media"]),
}
discrepancias = comprobar_numeros_permitidos(borrador, numeros_permitidos)
print("Discrepancias:", discrepancias if discrepancias else "ninguna")